## SRP029881 / GSE50726

**paper:** [PMID: 25176646](https://pmc.ncbi.nlm.nih.gov/articles/PMC4350764/) - Adaptations to a subterranean environment and longevity revealed by the analysis of mole rat genomes, 2015

**date, curator:** 2024-08-27, Sara Carsanaro

**notes**
* age and sex from SRA/GEO was used, as paper is very vague on these details
* extended methods call kit_name Illumina mRNA-Seq Prep Kit, so annotated as mRNA-seq Lib Prep Kit for Illumina. polyA is confirmed in extended methods

### annotation summary
run this after annotation is complete

In [29]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,kidney,UBERON:0002113,kidney,perfect match
1,ovary,UBERON:0000992,ovary,perfect match
3,testis,UBERON:0000473,testis,perfect match
4,liver,UBERON:0002107,liver,perfect match
5,frontal brain,UBERON:0016525,frontal lobe,perfect match
13,brain,UBERON:0000955,brain,perfect match


In [30]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,breeding,UBERON:0018241,prime adult stage,perfect match
1,3 years non-breeding,UBERON:0018241,prime adult stage,missing child term
3,5 years non-breeding,UBERON:0018241,prime adult stage,missing child term


### set variables, import packages, define functions

In [1]:
experiment_id = "SRP029881"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

/var/folders/b5/crkp117d43q5mcndnwlrww3w0000gn/T/ipykernel_16215/3311601719.py:3: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:11: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd
Be patient, it may take a few minutes.
0it [00:00, ?it/s]
0 samples dont have attributes, try to find them somewhere else
0it [00:

### library annnotations

In [6]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX347956,SRP029881,Illumina HiSeq 2000,SRS478733,,,,,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1227243,kidney,breeding,,,,F,,,885580,,,,,,BF_kidney,"SAMN02353297,GSM1227243",,,,,,,breeding female kidney,,,,27/08/2025,"For RNA-seq, poly-A+ RNA was isolated with 30 oligo-dT-coupled beads from 20 μg total RNA of each sample. First strand cDNA synthesis was performed with random hexamers and Superscript II reverse transcriptase (Invitrogen). The second strand was synthesized with E. coli DNA PolI (Invitrogen). Double stranded cDNA was purified with Qiaquick PCR purification kit (Qiagen), and sheared with a nebulizer (Invitrogen) to 100-500 bp fragments. After end repair and addition of a 3’ dA overhang the cDNA was ligated to Illumina PE adapter oligo mix, and size selected to 200±20 bp fragments by gel purification. After 15 cycles of PCR amplification the libraries were sequenced using Illumina HiSeq 2000 and the paired-end sequencing module.",,,,breeding female kidney,,,,TRANSCRIPTOMIC,cDNA,SRR975620,48B4CCA095209DA17860665724C7277A
1,SRX347955,SRP029881,Illumina HiSeq 2000,SRS478732,,,,,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1227242,ovary,3 years non-breeding,,,,F,,,885580,,,,,,3F_ovary,"SAMN02353298,GSM1227242",3,years,,,,,3 years old non-breeding female ovary,,,,27/08/2025,"For RNA-seq, poly-A+ RNA was isolated with 30 oligo-dT-coupled beads from 20 μg total RNA of each sample. First strand cDNA synthesis was performed with random hexamers and Superscript II reverse transcriptase (Invitrogen). The second strand was synthesized with E. coli DNA PolI (Invitrogen). Double stranded cDNA was purified with Qiaquick PCR purification kit (Qiagen), and sheared with a nebulizer (Invitrogen) to 100-500 bp fragments. After end repair and addition of a 3’ dA overhang the cDNA was ligated to Illumina PE adapter oligo mix, and size selected to 200±20 bp fragments by gel purification. After 15 cycles of PCR amplification the libraries were sequenced using Illumina HiSeq 2000 and the paired-end sequencing module.",,,,3 years old non-breeding female ovary,,,,TRANSCRIPTOMIC,cDNA,SRR975619,50AB85349B25C67528E7A995E419A0B3
2,SRX347954,SRP029881,Illumina HiSeq 2000,SRS478731,,,,,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1227241,ovary,breeding,,,,F,,,885580,,,,,,BF_ovary,"SAMN02353301,GSM1227241",,,,,,,breeding female ovary,,,,27/08/2025,"For RNA-seq, poly-A+ RNA was isolated with 30 oligo-dT-coupled beads from 20 μg total RNA of each sample. First strand cDNA synthesis was performed with random hexamers and Superscript II reverse transcriptase (Invitrogen). The second strand was synthesized with E. coli DNA PolI (Invitrogen). Double stranded cDNA was purified with Qiaquick PCR purification kit (Qiagen), and sheared with a nebulizer (Invitrogen) to 100-500 bp fragments. After end repair and addition of a 3’ dA overhang the cDNA was ligated to Illumina PE adapter oligo mix, and size selected to 200±20 bp fragments by gel purification. After 15 cycles of PCR amplification the libraries were sequenced using Illumina HiSeq 2000 and the paired-end sequencing module.",,,,breeding female ovary,,,,TRANSCRIPTOMIC,cDNA,SRR975618,A29D434E8C3FADF98849872573083282
3,SRX347953,SRP029881,Illumina HiSeq 2000,SRS478730,,,,,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1227240,testis,5 years non-breeding,,,,M,,,885580,,,,,,M5y_testis,"SAMN02353294,GSM1227240",5,

#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [5]:
unique_sorted(library, "infoOrgan")

['brain' 'frontal brain' 'kidney' 'liver' 'ovary' 'testis']


In [7]:

# brain
library.loc[library["infoOrgan"] == "brain", "anatId"] = "UBERON:0000955"
library.loc[library["infoOrgan"] == "brain", "anatName"] = "brain"
# perfect match, missing child term, other
library.loc[library["infoOrgan"] == "brain", "anatAnnotationStatus"] = "perfect match"
library.loc[library["infoOrgan"] == "brain", "anatBiologicalStatus"] = "not documented"

# frontal brain - per extended methods this is the frontal lobe region of the brain
library.loc[library["infoOrgan"] == "frontal brain", "anatId"] = "UBERON:0016525"
library.loc[library["infoOrgan"] == "frontal brain", "anatName"] = "frontal lobe"
# perfect match, missing child term, other
library.loc[library["infoOrgan"] == "frontal brain", "anatAnnotationStatus"] = "perfect match"
library.loc[library["infoOrgan"] == "frontal brain", "anatBiologicalStatus"] = "not documented"

# kidney
library.loc[library["infoOrgan"] == "kidney", "anatId"] = "UBERON:0002113"
library.loc[library["infoOrgan"] == "kidney", "anatName"] = "kidney"
# perfect match, missing child term, other
library.loc[library["infoOrgan"] == "kidney", "anatAnnotationStatus"] = "perfect match"
library.loc[library["infoOrgan"] == "kidney", "anatBiologicalStatus"] = "not documented"

# liver
library.loc[library["infoOrgan"] == "liver", "anatId"] = "UBERON:0002107"
library.loc[library["infoOrgan"] == "liver", "anatName"] = "liver"
# perfect match, missing child term, other
library.loc[library["infoOrgan"] == "liver", "anatAnnotationStatus"] = "perfect match"
library.loc[library["infoOrgan"] == "liver", "anatBiologicalStatus"] = "not documented"

# ovary
library.loc[library["infoOrgan"] == "ovary", "anatId"] = "UBERON:0000992"
library.loc[library["infoOrgan"] == "ovary", "anatName"] = "ovary"
# perfect match, missing child term, other
library.loc[library["infoOrgan"] == "ovary", "anatAnnotationStatus"] = "perfect match"
library.loc[library["infoOrgan"] == "ovary", "anatBiologicalStatus"] = "not documented"

# testis
library.loc[library["infoOrgan"] == "testis", "anatId"] = "UBERON:0000473"
library.loc[library["infoOrgan"] == "testis", "anatName"] = "testis"
# perfect match, missing child term, other
library.loc[library["infoOrgan"] == "testis", "anatAnnotationStatus"] = "perfect match"
library.loc[library["infoOrgan"] == "testis", "anatBiologicalStatus"] = "not documented"

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX347956,SRP029881,Illumina HiSeq 2000,SRS478733,UBERON:0002113,kidney,,,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1227243,kidney,breeding,perfect match,not documented,,F,,,885580,,,,,,BF_kidney,"SAMN02353297,GSM1227243",,,,,,,breeding female kidney,,,,27/08/2025,"For RNA-seq, poly-A+ RNA was isolated with 30 oligo-dT-coupled beads from 20 μg total RNA of each sample. First strand cDNA synthesis was performed with random hexamers and Superscript II reverse transcriptase (Invitrogen). The second strand was synthesized with E. coli DNA PolI (Invitrogen). Double stranded cDNA was purified with Qiaquick PCR purification kit (Qiagen), and sheared with a nebulizer (Invitrogen) to 100-500 bp fragments. After end repair and addition of a 3’ dA overhang the cDNA was ligated to Illumina PE adapter oligo mix, and size selected to 200±20 bp fragments by gel purification. After 15 cycles of PCR amplification the libraries were sequenced using Illumina HiSeq 2000 and the paired-end sequencing module.",,,,breeding female kidney,,,,TRANSCRIPTOMIC,cDNA,SRR975620,48B4CCA095209DA17860665724C7277A
1,SRX347955,SRP029881,Illumina HiSeq 2000,SRS478732,UBERON:0000992,ovary,,,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1227242,ovary,3 years non-breeding,perfect match,not documented,,F,,,885580,,,,,,3F_ovary,"SAMN02353298,GSM1227242",3,years,,,,,3 years old non-breeding female ovary,,,,27/08/2025,"For RNA-seq, poly-A+ RNA was isolated with 30 oligo-dT-coupled beads from 20 μg total RNA of each sample. First strand cDNA synthesis was performed with random hexamers and Superscript II reverse transcriptase (Invitrogen). The second strand was synthesized with E. coli DNA PolI (Invitrogen). Double stranded cDNA was purified with Qiaquick PCR purification kit (Qiagen), and sheared with a nebulizer (Invitrogen) to 100-500 bp fragments. After end repair and addition of a 3’ dA overhang the cDNA was ligated to Illumina PE adapter oligo mix, and size selected to 200±20 bp fragments by gel purification. After 15 cycles of PCR amplification the libraries were sequenced using Illumina HiSeq 2000 and the paired-end sequencing module.",,,,3 years old non-breeding female ovary,,,,TRANSCRIPTOMIC,cDNA,SRR975619,50AB85349B25C67528E7A995E419A0B3
2,SRX347954,SRP029881,Illumina HiSeq 2000,SRS478731,UBERON:0000992,ovary,,,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1227241,ovary,breeding,perfect match,not documented,,F,,,885580,,,,,,BF_ovary,"SAMN02353301,GSM1227241",,,,,,,breeding female ovary,,,,27/08/2025,"For RNA-seq, poly-A+ RNA was isolated with 30 oligo-dT-coupled beads from 20 μg total RNA of each sample. First strand cDNA synthesis was performed with random hexamers and Superscript II reverse transcriptase (Invitrogen). The second strand was synthesized with E. coli DNA PolI (Invitrogen). Double stranded cDNA was purified with Qiaquick PCR purification kit (Qiagen), and sheared with a nebulizer (Invitrogen) to 100-500 bp fragments. After end repair and addition of a 3’ dA overhang the cDNA was ligated to Illumina PE adapter oligo mix, and size selected to 200±20 bp fragments by gel purification. After 15 cycles of PCR amplification the libraries were sequenced using Illumina HiSeq 2000 and the paired-end sequencing module.",,,,breeding female ovary,,,,TRANSCRIPTOMIC,cDNA,SRR975618,A29D434E8C3FADF98849872573083282
3,SRX347953,SRP029881,Illumina HiSeq 2000,SRS478730,UBERON:00004

#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src)

In [8]:
unique_sorted(library, "infoStage")

['3 years non-breeding' '5 years  non-breeding' 'breeding']


In [9]:
# 3 years non-breeding
library.loc[library["infoStage"] == "3 years non-breeding", "stageId"] = "UBERON:0018241"
library.loc[library["infoStage"] == "3 years non-breeding", "stageName"] = "prime adult stage"
# perfect match, missing child term, other
library.loc[library["infoStage"] == "3 years non-breeding", "stageAnnotationStatus"] = "missing child term"

# 5 years  non-breeding
library.loc[library["infoStage"] == "5 years  non-breeding", "stageId"] = "UBERON:0018241"
library.loc[library["infoStage"] == "5 years  non-breeding", "stageName"] = "prime adult stage"
# perfect match, missing child term, other
library.loc[library["infoStage"] == "5 years  non-breeding", "stageAnnotationStatus"] = "missing child term"

# breeding
library.loc[library["infoStage"] == "breeding", "stageId"] = "UBERON:0018241"
library.loc[library["infoStage"] == "breeding", "stageName"] = "prime adult stage"
# perfect match, missing child term, other
library.loc[library["infoStage"] == "breeding", "stageAnnotationStatus"] = "perfect match"

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX347956,SRP029881,Illumina HiSeq 2000,SRS478733,UBERON:0002113,kidney,UBERON:0018241,prime adult stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1227243,kidney,breeding,perfect match,not documented,perfect match,F,,,885580,,,,,,BF_kidney,"SAMN02353297,GSM1227243",,,,,,,breeding female kidney,,,,27/08/2025,"For RNA-seq, poly-A+ RNA was isolated with 30 oligo-dT-coupled beads from 20 μg total RNA of each sample. First strand cDNA synthesis was performed with random hexamers and Superscript II reverse transcriptase (Invitrogen). The second strand was synthesized with E. coli DNA PolI (Invitrogen). Double stranded cDNA was purified with Qiaquick PCR purification kit (Qiagen), and sheared with a nebulizer (Invitrogen) to 100-500 bp fragments. After end repair and addition of a 3’ dA overhang the cDNA was ligated to Illumina PE adapter oligo mix, and size selected to 200±20 bp fragments by gel purification. After 15 cycles of PCR amplification the libraries were sequenced using Illumina HiSeq 2000 and the paired-end sequencing module.",,,,breeding female kidney,,,,TRANSCRIPTOMIC,cDNA,SRR975620,48B4CCA095209DA17860665724C7277A
1,SRX347955,SRP029881,Illumina HiSeq 2000,SRS478732,UBERON:0000992,ovary,UBERON:0018241,prime adult stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1227242,ovary,3 years non-breeding,perfect match,not documented,missing child term,F,,,885580,,,,,,3F_ovary,"SAMN02353298,GSM1227242",3,years,,,,,3 years old non-breeding female ovary,,,,27/08/2025,"For RNA-seq, poly-A+ RNA was isolated with 30 oligo-dT-coupled beads from 20 μg total RNA of each sample. First strand cDNA synthesis was performed with random hexamers and Superscript II reverse transcriptase (Invitrogen). The second strand was synthesized with E. coli DNA PolI (Invitrogen). Double stranded cDNA was purified with Qiaquick PCR purification kit (Qiagen), and sheared with a nebulizer (Invitrogen) to 100-500 bp fragments. After end repair and addition of a 3’ dA overhang the cDNA was ligated to Illumina PE adapter oligo mix, and size selected to 200±20 bp fragments by gel purification. After 15 cycles of PCR amplification the libraries were sequenced using Illumina HiSeq 2000 and the paired-end sequencing module.",,,,3 years old non-breeding female ovary,,,,TRANSCRIPTOMIC,cDNA,SRR975619,50AB85349B25C67528E7A995E419A0B3
2,SRX347954,SRP029881,Illumina HiSeq 2000,SRS478731,UBERON:0000992,ovary,UBERON:0018241,prime adult stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1227241,ovary,breeding,perfect match,not documented,perfect match,F,,,885580,,,,,,BF_ovary,"SAMN02353301,GSM1227241",,,,,,,breeding female ovary,,,,27/08/2025,"For RNA-seq, poly-A+ RNA was isolated with 30 oligo-dT-coupled beads from 20 μg total RNA of each sample. First strand cDNA synthesis was performed with random hexamers and Superscript II reverse transcriptase (Invitrogen). The second strand was synthesized with E. coli DNA PolI (Invitrogen). Double stranded cDNA was purified with Qiaquick PCR purification kit (Qiagen), and sheared with a nebulizer (Invitrogen) to 100-500 bp fragments. After end repair and addition of a 3’ dA overhang the cDNA was ligated to Illumina PE adapter oligo mix, and size selected to 200±20 bp fragments by gel purification. After 15 cycles of PCR amplification the libraries were sequenced using Illumina HiSeq 2000 and the paired-end sequencing module.",,,,breeding female

#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [ ]:
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [10]:
# making these variables because we use them again in the experiment file
my_protocol = 'mRNA-seq Lib Prep Kit for Illumina'
# full_length or 3'
my_protocolType = 'full_length'

library.loc[:,'protocol'] = my_protocol
library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'polyA'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX347956,SRP029881,Illumina HiSeq 2000,SRS478733,UBERON:0002113,kidney,UBERON:0018241,prime adult stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1227243,kidney,breeding,perfect match,not documented,perfect match,F,,,885580,mRNA-seq Lib Prep Kit for Illumina,full_length,polyA,,,BF_kidney,"SAMN02353297,GSM1227243",,,,,,,breeding female kidney,,,,27/08/2025,"For RNA-seq, poly-A+ RNA was isolated with 30 oligo-dT-coupled beads from 20 μg total RNA of each sample. First strand cDNA synthesis was performed with random hexamers and Superscript II reverse transcriptase (Invitrogen). The second strand was synthesized with E. coli DNA PolI (Invitrogen). Double stranded cDNA was purified with Qiaquick PCR purification kit (Qiagen), and sheared with a nebulizer (Invitrogen) to 100-500 bp fragments. After end repair and addition of a 3’ dA overhang the cDNA was ligated to Illumina PE adapter oligo mix, and size selected to 200±20 bp fragments by gel purification. After 15 cycles of PCR amplification the libraries were sequenced using Illumina HiSeq 2000 and the paired-end sequencing module.",,,,breeding female kidney,,,,TRANSCRIPTOMIC,cDNA,SRR975620,48B4CCA095209DA17860665724C7277A
1,SRX347955,SRP029881,Illumina HiSeq 2000,SRS478732,UBERON:0000992,ovary,UBERON:0018241,prime adult stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1227242,ovary,3 years non-breeding,perfect match,not documented,missing child term,F,,,885580,mRNA-seq Lib Prep Kit for Illumina,full_length,polyA,,,3F_ovary,"SAMN02353298,GSM1227242",3,years,,,,,3 years old non-breeding female ovary,,,,27/08/2025,"For RNA-seq, poly-A+ RNA was isolated with 30 oligo-dT-coupled beads from 20 μg total RNA of each sample. First strand cDNA synthesis was performed with random hexamers and Superscript II reverse transcriptase (Invitrogen). The second strand was synthesized with E. coli DNA PolI (Invitrogen). Double stranded cDNA was purified with Qiaquick PCR purification kit (Qiagen), and sheared with a nebulizer (Invitrogen) to 100-500 bp fragments. After end repair and addition of a 3’ dA overhang the cDNA was ligated to Illumina PE adapter oligo mix, and size selected to 200±20 bp fragments by gel purification. After 15 cycles of PCR amplification the libraries were sequenced using Illumina HiSeq 2000 and the paired-end sequencing module.",,,,3 years old non-breeding female ovary,,,,TRANSCRIPTOMIC,cDNA,SRR975619,50AB85349B25C67528E7A995E419A0B3
2,SRX347954,SRP029881,Illumina HiSeq 2000,SRS478731,UBERON:0000992,ovary,UBERON:0018241,prime adult stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1227241,ovary,breeding,perfect match,not documented,perfect match,F,,,885580,mRNA-seq Lib Prep Kit for Illumina,full_length,polyA,,,BF_ovary,"SAMN02353301,GSM1227241",,,,,,,breeding female ovary,,,,27/08/2025,"For RNA-seq, poly-A+ RNA was isolated with 30 oligo-dT-coupled beads from 20 μg total RNA of each sample. First strand cDNA synthesis was performed with random hexamers and Superscript II reverse transcriptase (Invitrogen). The second strand was synthesized with E. coli DNA PolI (Invitrogen). Double stranded cDNA was purified with Qiaquick PCR purification kit (Qiagen), and sheared with a nebulizer (Invitrogen) to 100-500 bp fragments. After end repair and addition of a 3’ dA overhang the cDNA was ligated to Illumina PE adapter oligo mix, and size selected to 200±20 bp fragments by gel purification.

#### globin, replicates

In [11]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

no duplicate values in SRSId


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status

In [ ]:
#library.loc[:,'sampleAge_value'] = ''
#library.loc[:,'sampleAge_unit'] = ''

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# view
display_df(library)

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [23]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2024-08-27'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX347956,SRP029881,Illumina HiSeq 2000,SRS478733,UBERON:0002113,kidney,UBERON:0018241,prime adult stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1227243,kidney,breeding,perfect match,not documented,perfect match,F,,,885580,mRNA-seq Lib Prep Kit for Illumina,full_length,polyA,,,BF_kidney,"SAMN02353297,GSM1227243",,,,,,,breeding female kidney,,breeding,SAC,2024-08-27
1,SRX347955,SRP029881,Illumina HiSeq 2000,SRS478732,UBERON:0000992,ovary,UBERON:0018241,prime adult stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1227242,ovary,3 years non-breeding,perfect match,not documented,missing child term,F,,,885580,mRNA-seq Lib Prep Kit for Illumina,full_length,polyA,,,3F_ovary,"SAMN02353298,GSM1227242",3,years,,,,,3 years old non-breeding female ovary,,non-breeding,SAC,2024-08-27
2,SRX347954,SRP029881,Illumina HiSeq 2000,SRS478731,UBERON:0000992,ovary,UBERON:0018241,prime adult stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1227241,ovary,breeding,perfect match,not documented,perfect match,F,,,885580,mRNA-seq Lib Prep Kit for Illumina,full_length,polyA,,,BF_ovary,"SAMN02353301,GSM1227241",,,,,,,breeding female ovary,,breeding,SAC,2024-08-27
3,SRX347953,SRP029881,Illumina HiSeq 2000,SRS478730,UBERON:0000473,testis,UBERON:0018241,prime adult stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1227240,testis,5 years non-breeding,perfect match,not documented,missing child term,M,,,885580,mRNA-seq Lib Prep Kit for Illumina,full_length,polyA,,,M5y_testis,"SAMN02353294,GSM1227240",5,years,,,,,5 years old non-breeding male testis,,non-breeding,SAC,2024-08-27
4,SRX347952,SRP029881,Illumina HiSeq 2000,SRS478729,UBERON:0002107,liver,UBERON:0018241,prime adult stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1227239,liver,breeding,perfect match,not documented,perfect match,M,,,885580,mRNA-seq Lib Prep Kit for Illumina,full_length,polyA,,,BM_liver,"SAMN02353299,GSM1227239",,,,,,,breeding male liver,,breeding,SAC,2024-08-27
5,SRX347951,SRP029881,Illumina HiSeq 2000,SRS478728,UBERON:0016525,frontal lobe,UBERON:0018241,prime adult stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1227238,frontal brain,breeding,perfect match,not documented,perfect match,M,,,885580,mRNA-seq Lib Prep Kit for Illumina,full_length,polyA,,,BM_brain_front,"SAMN02353300,GSM1227238",,,,,,,breeding male frontal brain,,breeding,SAC,2024-08-27
6,SRX347950,SRP029881,Illumina HiSeq 2000,SRS478727,UBERON:0016525,frontal lobe,UBERON:0018241,prime adult stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1227237,frontal brain,breeding,perfect match,not documented,perfect match,F,,,885580,mRNA-seq Lib Prep Kit for Illumina,full_length,polyA,,,BF_brain_front,"SAMN02353292,GSM1227237",,,,,,,breeding female frontal brain,,breeding,SAC,2024-08-27
7,SRX347949,SRP029881,Illumina HiSeq 2000,SRS478726,UBERON:0002107,liver,UBERON:0018241,prime adult stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1227236,liver,3 years non-breeding,perfect match,not documented,missing child term,F,,,885580,mRNA-seq Lib Prep Kit for Illumina,full_length,polyA,,,3F_liver,"SAMN02353293,GSM1227236",3,years,,,,,3 years old non-breeding female liver,,non-breeding,SAC,2024-08-27
8,SRX347948,SRP029881,Illumina HiSeq 2000,SRS478725,UBERON:0000473,testis,UBERON:0018241,prime adult stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1227235,testis,breeding,perfect match,not documented,perfect match,M,,,885580,mRNA-seq Lib Prep Kit for Illumina,full_length,polyA,,,BM_testis,"SAMN02353289,GSM1227235",,,,,,,breeding male testis,,breeding,SA

#### comments

In [ ]:
#library.loc[:,'comment'] = ''

#### save complete file with correct columns

In [24]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX347956,SRP029881,Illumina HiSeq 2000,SRS478733,UBERON:0002113,kidney,UBERON:0018241,prime adult stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1227243,kidney,breeding,perfect match,not documented,perfect match,F,,,885580,mRNA-seq Lib Prep Kit for Illumina,full_length,polyA,,,BF_kidney,"SAMN02353297,GSM1227243",,,,,,,breeding female kidney,,breeding,SAC,2024-08-27
1,SRX347955,SRP029881,Illumina HiSeq 2000,SRS478732,UBERON:0000992,ovary,UBERON:0018241,prime adult stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1227242,ovary,3 years non-breeding,perfect match,not documented,missing child term,F,,,885580,mRNA-seq Lib Prep Kit for Illumina,full_length,polyA,,,3F_ovary,"SAMN02353298,GSM1227242",3,years,,,,,3 years old non-breeding female ovary,,non-breeding,SAC,2024-08-27
2,SRX347954,SRP029881,Illumina HiSeq 2000,SRS478731,UBERON:0000992,ovary,UBERON:0018241,prime adult stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1227241,ovary,breeding,perfect match,not documented,perfect match,F,,,885580,mRNA-seq Lib Prep Kit for Illumina,full_length,polyA,,,BF_ovary,"SAMN02353301,GSM1227241",,,,,,,breeding female ovary,,breeding,SAC,2024-08-27
3,SRX347953,SRP029881,Illumina HiSeq 2000,SRS478730,UBERON:0000473,testis,UBERON:0018241,prime adult stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1227240,testis,5 years non-breeding,perfect match,not documented,missing child term,M,,,885580,mRNA-seq Lib Prep Kit for Illumina,full_length,polyA,,,M5y_testis,"SAMN02353294,GSM1227240",5,years,,,,,5 years old non-breeding male testis,,non-breeding,SAC,2024-08-27
4,SRX347952,SRP029881,Illumina HiSeq 2000,SRS478729,UBERON:0002107,liver,UBERON:0018241,prime adult stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1227239,liver,breeding,perfect match,not documented,perfect match,M,,,885580,mRNA-seq Lib Prep Kit for Illumina,full_length,polyA,,,BM_liver,"SAMN02353299,GSM1227239",,,,,,,breeding male liver,,breeding,SAC,2024-08-27
5,SRX347951,SRP029881,Illumina HiSeq 2000,SRS478728,UBERON:0016525,frontal lobe,UBERON:0018241,prime adult stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1227238,frontal brain,breeding,perfect match,not documented,perfect match,M,,,885580,mRNA-seq Lib Prep Kit for Illumina,full_length,polyA,,,BM_brain_front,"SAMN02353300,GSM1227238",,,,,,,breeding male frontal brain,,breeding,SAC,2024-08-27
6,SRX347950,SRP029881,Illumina HiSeq 2000,SRS478727,UBERON:0016525,frontal lobe,UBERON:0018241,prime adult stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1227237,frontal brain,breeding,perfect match,not documented,perfect match,F,,,885580,mRNA-seq Lib Prep Kit for Illumina,full_length,polyA,,,BF_brain_front,"SAMN02353292,GSM1227237",,,,,,,breeding female frontal brain,,breeding,SAC,2024-08-27
7,SRX347949,SRP029881,Illumina HiSeq 2000,SRS478726,UBERON:0002107,liver,UBERON:0018241,prime adult stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1227236,liver,3 years non-breeding,perfect match,not documented,missing child term,F,,,885580,mRNA-seq Lib Prep Kit for Illumina,full_length,polyA,,,3F_liver,"SAMN02353293,GSM1227236",3,years,,,,,3 years old non-breeding female liver,,non-breeding,SAC,2024-08-27
8,SRX347948,SRP029881,Illumina HiSeq 2000,SRS478725,UBERON:0000473,testis,UBERON:0018241,prime adult stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM1227235,testis,breeding,perfect match,not documented,perfect match,M,,,885580,mRNA-seq Lib Prep Kit for Illumina,full_length,polyA,,,BM_testis,"SAMN02353289,GSM1227235",,,,,,,breeding male testis,,breeding,SA

### experiment annotations

In [14]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP029881,Transcriptome sequencing of Fukomys damarensis,Deep sequencing of mRNA from Fukomys damarensis to uncover the reproduce regulation Overall design: Analysis of ploy(A)+ RNA of Fukomys damarensis,SRA,,,,,,GSE50726,PRJNA218853,25176646,,10.1016/j.celrep.2014.07.030,,


#### experiment and protocol details

In [15]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

16

In [16]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'total'
#experiment.loc[:,'projectTags'] = '' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
experiment.loc[:,'protocol'] = my_protocol
experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP029881,Transcriptome sequencing of Fukomys damarensis,Deep sequencing of mRNA from Fukomys damarensis to uncover the reproduce regulation Overall design: Analysis of ploy(A)+ RNA of Fukomys damarensis,SRA,total,,16,mRNA-seq Lib Prep Kit for Illumina,full_length,GSE50726,PRJNA218853,25176646,,10.1016/j.celrep.2014.07.030,,


#### paper and xrefs

In [17]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
#experiment.loc[:,'PMID'] = ''
experiment.loc[:,'reference_url'] = 'https://pmc.ncbi.nlm.nih.gov/articles/PMC4350764/'
#experiment.loc[:,'DOI'] = ''
#experiment.loc[:,'xrefs'] = ''

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP029881,Transcriptome sequencing of Fukomys damarensis,Deep sequencing of mRNA from Fukomys damarensis to uncover the reproduce regulation Overall design: Analysis of ploy(A)+ RNA of Fukomys damarensis,SRA,total,,16,mRNA-seq Lib Prep Kit for Illumina,full_length,GSE50726,PRJNA218853,25176646,https://pmc.ncbi.nlm.nih.gov/articles/PMC4350764/,10.1016/j.celrep.2014.07.030,,


#### comments

In [ ]:
#experiment.loc[:,'comment'] = ''

display_df(experiment)

#### save complete file

In [18]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [25]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

#### to add things here

#### check columns match

In [26]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [27]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
48804,SRX28039819,SRP571305,Illumina HiSeq X Ten,SRS24400533,UBERON:0000002,uterine cervix,OariDv:0000007,adulthood stage,,cervix,2 years,perfect match,not documented,missing child term,F,Small-Tailed Han Sheep,,9940,Stranded Total RNA,full_length,ribo-minus,,,RNASeq_cervix_40,SAMN47422521,,,,,,,"PMID:40308692, The ewes were 3 years old, sexu...",,,ANN,2025-08-18
48805,SRX28039818,SRP571305,Illumina HiSeq X Ten,SRS24400531,UBERON:0000002,uterine cervix,OariDv:0000007,adulthood stage,,cervix,2 years,perfect match,not documented,missing child term,F,Small-Tailed Han Sheep,,9940,Stranded Total RNA,full_length,ribo-minus,,,RNASeq_cervix_39,SAMN47422520,,,,,,,"PMID:40308692, The ewes were 3 years old, sexu...",,,ANN,2025-08-18
48806,SRX347956,SRP029881,Illumina HiSeq 2000,SRS478733,UBERON:0002113,kidney,UBERON:0018241,prime adult stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?...,kidney,breeding,perfect match,not documented,perfect match,F,,,885580,mRNA-seq Lib Prep Kit for Illumina,full_length,polyA,,,BF_kidney,"SAMN02353297,GSM1227243",,,,,,,breeding female kidney,,breeding,SAC,2024-08-27
48807,SRX347955,SRP029881,Illumina HiSeq 2000,SRS478732,UBERON:0000992,ovary,UBERON:0018241,prime adult stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?...,ovary,3 years non-breeding,perfect match,not documented,missing child term,F,,,885580,mRNA-seq Lib Prep Kit for Illumina,full_length,polyA,,,3F_ovary,"SAMN02353298,GSM1227242",3,years,,,,,3 years old non-breeding female ovary,,non-breeding,SAC,2024-08-27
48808,SRX347954,SRP029881,Illumina HiSeq 2000,SRS478731,UBERON:0000992,ovary,UBERON:0018241,prime adult stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?...,ovary,breeding,perfect match,not documented,perfect match,F,,,885580,mRNA-seq Lib Prep Kit for Illumina,full_length,polyA,,,BF_ovary,"SAMN02353301,GSM1227241",,,,,,,breeding female ovary,,breeding,SAC,2024-08-27
48809,SRX347953,SRP029881,Illumina HiSeq 2000,SRS478730,UBERON:0000473,testis,UBERON:0018241,prime adult stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?...,testis,5 years non-breeding,perfect match,not documented,missing child term,M,,,885580,mRNA-seq Lib Prep Kit for Illumina,full_length,polyA,,,M5y_testis,"SAMN02353294,GSM1227240",5,years,,,,,5 years old non-breeding male testis,,non-breeding,SAC,2024-08-27
48810,SRX347952,SRP029881,Illumina HiSeq 2000,SRS478729,UBERON:0002107,liver,UBERON:0018241,prime adult stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?...,liver,breeding,perfect match,not documented,perfect match,M,,,885580,mRNA-seq Lib Prep Kit for Illumina,full_length,polyA,,,BM_liver,"SAMN02353299,GSM1227239",,,,,,,breeding male liver,,breeding,SAC,2024-08-27


In [28]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
913,#SRP581857,Influence of pregnancy on the mammary gland of...,Kazakh mares have drawn significant attention ...,SRA,total,,16,,,,PRJNA1256214,40723519,https://pmc.ncbi.nlm.nih.gov/articles/PMC12291...,10.3390/ani15142056,,Rejected by Sagane D: methods for RNA types ar...
914,SRP571305,Construction of a Regulatory Element Map in Sheep,Comprehensive functional genome annotation is ...,SRA,partial,FAANG,10,Stranded Total RNA,full_length,,PRJNA1237432,40308692,https://pmc.ncbi.nlm.nih.gov/articles/PMC12040...,10.3389/fvets.2025.1564148,,Info from SRA and from publication are differe...
915,SRP029881,Transcriptome sequencing of Fukomys damarensis,Deep sequencing of mRNA from Fukomys damarensi...,SRA,total,,16,mRNA-seq Lib Prep Kit for Illumina,full_length,GSE50726,PRJNA218853,25176646,https://pmc.ncbi.nlm.nih.gov/articles/PMC4350764/,10.1016/j.celrep.2014.07.030,,


### add annotations to git

In [ ]:
! git pull

In [ ]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [ ]:
! git status

In [ ]:
! git add $git_experiment_path $git_library_path

In [ ]:
! git commit -m $commit_message_exp

In [ ]:
! git push

### add annotation folder and script to git

In [ ]:
! git status

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push